# Real-Time Object Detection using TensorFlow & OpenCV

In [ ]:
import numpy as np
import os
import sys
import tensorflow as tf

# Ensure TensorFlow 2.x compatibility for legacy gfile calls
if not hasattr(tf, 'gfile'):
    tf.gfile = tf.io.gfile

## Environment Setup
Add `models/research` and `models/research/object_detection` to system path so utility modules can be resolved.

In [ ]:
CWD_PATH = os.getcwd()
RESEARCH_PATH = os.path.join(CWD_PATH, 'models', 'research')
OBJECT_DETECTION_PATH = os.path.join(RESEARCH_PATH, 'object_detection')

for path in [RESEARCH_PATH, OBJECT_DETECTION_PATH]:
    if path not in sys.path:
        sys.path.insert(0, path)

## Object Detection API Imports

In [ ]:
from object_detection.utils import label_map_util
from object_detection.utils import visualization_utils as vis_util

## Model & Label Map Setup

In [ ]:
MODEL_NAME = 'ssd_mobilenet_v1_coco_2017_11_17'
PATH_TO_CKPT = os.path.join(CWD_PATH, MODEL_NAME, 'frozen_inference_graph.pb')
PATH_TO_LABELS = os.path.join(OBJECT_DETECTION_PATH, 'data', 'mscoco_label_map.pbtxt')
NUM_CLASSES = 90

## Load Frozen TensorFlow Model Graph

In [ ]:
detection_graph = tf.Graph()
with detection_graph.as_default():
    od_graph_def = tf.compat.v1.GraphDef()
    with tf.io.gfile.GFile(PATH_TO_CKPT, 'rb') as fid:
        serialized_graph = fid.read()
        od_graph_def.ParseFromString(serialized_graph)
        tf.import_graph_def(od_graph_def, name='')

print('Model loaded successfully into memory.')

## Load Label Map & Category Index

In [ ]:
label_map = label_map_util.load_labelmap(PATH_TO_LABELS)
categories = label_map_util.convert_label_map_to_categories(label_map, max_num_classes=NUM_CLASSES, use_display_name=True)
category_index = label_map_util.create_category_index(categories)

print('Label map loaded successfully.')

## Real-Time Webcam Detection Loop

In [ ]:
import cv2

# Open default webcam (device 0)
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print('Error: Could not open webcam. Ensure camera is connected and permissions are granted.')
else:
    print("Starting live detection. Press 'q' in the window to stop.")
    with detection_graph.as_default():
        with tf.compat.v1.Session(graph=detection_graph) as sess:
            image_tensor = detection_graph.get_tensor_by_name('image_tensor:0')
            detection_boxes = detection_graph.get_tensor_by_name('detection_boxes:0')
            detection_scores = detection_graph.get_tensor_by_name('detection_scores:0')
            detection_classes = detection_graph.get_tensor_by_name('detection_classes:0')
            num_detections = detection_graph.get_tensor_by_name('num_detections:0')

            try:
                while cap.isOpened():
                    ret, image_np = cap.read()
                    if not ret or image_np is None:
                        print('Failed to grab frame from webcam.')
                        break

                    # Expand image dimensions for network input: [1, H, W, C]
                    image_np_expanded = np.expand_dims(image_np, axis=0)

                    # Perform object detection
                    (boxes, scores, classes, num) = sess.run(
                        [detection_boxes, detection_scores, detection_classes, num_detections],
                        feed_dict={image_tensor: image_np_expanded}
                    )

                    # Draw detected boxes and labels
                    vis_util.visualize_boxes_and_labels_on_image_array(
                        image_np,
                        np.squeeze(boxes),
                        np.squeeze(classes).astype(np.int32),
                        np.squeeze(scores),
                        category_index,
                        use_normalized_coordinates=True,
                        line_thickness=8
                    )

                    # Display live window
                    cv2.imshow('Real-Time Object Detection', image_np)

                    # Exit on 'q'
                    if cv2.waitKey(1) & 0xFF == ord('q'):
                        print('User requested exit.')
                        break
            finally:
                cap.release()
                cv2.destroyAllWindows()
                print('Webcam stream stopped and windows closed cleanly.')